In [ ]:
# Open this notebook in the repository containing the three YAML files.
# In Colab, upload/mount that working copy and change into it first.
from pathlib import Path
assert Path('inference.yml').is_file(), 'Change into the migrated Portuguese-IFEval repository first.'
%pip install -q -r requirements.txt
from config import load_config, project_path


Use the versioned `universal_inference.py` and the three YAML files. This notebook no longer overwrites repository scripts.


In [ ]:
# Download only the NLP resources configured for evaluation.
import subprocess
import sys
import nltk
nlp_config = load_config('metrics')['nlp']
for key in ('portuguese_model', 'spanish_model', 'multilingual_sentence_model'):
    subprocess.run([sys.executable, '-m', 'spacy', 'download', nlp_config[key]], check=True)
nltk.download(nlp_config['nltk_resource'])


In [ ]:
# Validate canonical inputs; do not rename or alter the curated dataset.
import json
io_config = load_config('inference')['io']
for language, value in io_config['input_files'].items():
    path = project_path(value)
    with path.open(encoding='utf-8') as stream:
        print(language, sum(bool(line.strip()) for line in stream), path)


The canonical PT dataset is used unchanged. Historical filtering is not rerun by this notebook.

In [ ]:
# Review selected models and settings before starting a GPU/API run.
inference_config = load_config('inference')
print(inference_config['benchmark_runner']['models_to_benchmark'])
print(inference_config['universal_inference'])


Portuguese sentence tokenization uses the versioned utility and the NLP resources in `metrics.yml`.

Use the versioned `instruction_utils/pt_instructions_util.py` and the three YAML files. This notebook no longer overwrites repository scripts.


In [ ]:
# Configure inference.yml and metrics.yml before launching.
!python benchmark_runner.py --config inference.yml --metrics-config metrics.yml --datasets pt --dry-run
# Remove --dry-run only after reviewing models, resources and output paths.


In [ ]:
# Download generated responses from the configured output directory (Colab).
from google.colab import files
import tempfile
import zipfile
io_config = load_config('inference')['io']
responses = project_path(io_config['responses_dir'])
with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as temp:
    archive_path = temp.name
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(responses.glob('*' + io_config['response_extension'])):
        archive.write(path, path.name)
files.download(archive_path)


In [ ]:
# Download generated evaluations from the configured output directory (Colab).
from google.colab import files
import tempfile
import zipfile
evaluations = project_path(load_config('metrics')['paths']['generated_evaluations'])
with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as temp:
    archive_path = temp.name
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(evaluations.rglob('*.jsonl')):
        archive.write(path, path.relative_to(evaluations))
files.download(archive_path)
